In [1]:
import time
import pyautogui as pyg

pyg.sleep(2)

# 自動偵測 Mac Retina 螢幕縮放倍率 (1x 或 2x)
screen_w, _ = pyg.size()
test_im = pyg.screenshot()
scale = round(test_im.width / screen_w)
print(f"目前螢幕解析度倍率為: {scale}x")

basex = 542
basey = 425
gap = 40

目前螢幕解析度倍率為: 2x


In [2]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    # 關鍵修正：必須乘以 scale 才能抓到 Retina 螢幕上的真實寶石顏色
    pixel = im.getpixel((x * scale, y * scale))
    return pixel[:3]

def is_same_color(c1, c2, tolerance=25):
    if c1 is None or c2 is None:
        return False
    # 過濾背景：RGB 總和小於 150 (太暗) 或全黑/深紫背景，一律不視為有效寶石
    if sum(c1) < 150 or sum(c2) < 150:
        return False
    return (abs(c1[0] - c2[0]) + abs(c1[1] - c2[1]) + abs(c1[2] - c2[2])) <= tolerance

def _can_swap_and_match(r, c, dr, dc, im):
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    # 如果顏色太暗（背景）或兩顆寶石顏色本來就一樣，不處理
    if sum(current_color) < 150 or sum(neighbor_color) < 150:
        return False
    if is_same_color(current_color, neighbor_color, tolerance=10):
        return False

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        if sum(color) < 150: # 背景不參與連線計算
            return False

        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and is_same_color(color_after_swap(check_r, check_c), color)):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_left(r, c, im): return _can_swap_and_match(r, c, 0, -1, im)
def can_move_right(r, c, im): return _can_swap_and_match(r, c, 0, 1, im)
def can_move_up(r, c, im): return _can_swap_and_match(r, c, -1, 0, im)
def can_move_down(r, c, im): return _can_swap_and_match(r, c, 1, 0, im)

In [3]:
import time
import pyautogui as pyg

# 1. 【最關鍵】將 PyAutoGUI 預設點擊延遲降至最低 (預設為 0.1 秒)
pyg.PAUSE = 0.001

start_time = time.time()
im = pyg.screenshot()

last_move = None
repeat_count = 0

# 2. 只需檢查「向右」與「向下」，即可涵蓋全盤所有交換組合 (避免重複計算)
directions = (
    (can_move_right, 0, 1),
    (can_move_down, 1, 0),
)

while time.time() - start_time <= 70:
    moved = False

    for r in range(8):
        for c in range(8):
            for can_move, dr, dc in directions:
                # 邊界保護：避免超出 8x8 盤面
                if r + dr >= 8 or c + dc >= 8:
                    continue

                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    current_move = (x1, y1, x2, y2)

                    # 強制防卡死機制：同一組合連續嘗試 2 次以上跳過
                    if current_move == last_move:
                        repeat_count += 1
                        if repeat_count >= 2:
                            continue
                    else:
                        repeat_count = 0

                    last_move = current_move

                    # 3. 極速點擊交換
                    pyg.click(x1, y1)
                    pyg.sleep(0.01)  # 點擊間隔縮短至 0.01 秒
                    pyg.click(x2, y2)

                    # 4. 擷取新畫面
                    pyg.sleep(0.01)
                    im = pyg.screenshot()
                    moved = True
                    break

            if moved:
                break

        if moved:
            break

    if not moved:
        last_move = None
        repeat_count = 0
        im = pyg.screenshot()

print("已執行滿 70 秒，自動停止")

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.